# LangChain + ChromaDB: RAG básico

El objetivo es construir un **RAG 2-step** con LangChain v1:

- **OpenAI** para chat y embeddings, via `init_chat_model` e `init_embeddings`.
- **Chroma** (`langchain-chroma`) como vector store local persistente.
- Recuperación explícita y después generación: `retriever.invoke()` y luego `llm.invoke()`.

> Idea central: un LLM no conoce automáticamente tus documentos privados. RAG busca fragmentos relevantes en una base vectorial y se los pasa al modelo como contexto.
>
> **LCEL** (`prompt | modelo | parser`) sigue siendo útil para el paso lineal de generación. Para una app conversacional que decide *si* buscar, el siguiente paso es `create_agent`.


## 0. Qué vamos a construir

Al final del notebook tendremos esta arquitectura de **RAG 2-step**:

```text
Documentos -> chunks -> embeddings -> Chroma
                                    |
Pregunta del usuario -> retriever --+
                                    |
                          prompt con contexto -> chat model -> respuesta
```

La recuperación ocurre *siempre* antes de generar. Eso es predecible y fácil de debuggear en clase.

El ejemplo usa documentos creados dentro del notebook para que funcione sin archivos externos. Después hay una variante con PDFs (`pypdf`) y un puente a **RAG agentic** con `create_agent`.


## 1. Instalación

Ejecutar esta celda una vez en un entorno limpio.

Paquetes:

- `langchain[openai]`: agentes, `init_chat_model`, `init_embeddings` y la integración con OpenAI.
- `langchain-chroma`: integración actual entre LangChain y Chroma (`chromadb` viene como dependencia).
- `langchain-text-splitters`: splitters de documentos.
- `pypdf`: carga de PDFs al final del notebook. No usamos `langchain-community` (ese paquete está sunset).
- `python-dotenv`: cargar un archivo `.env`.


In [ ]:
%pip install -U "langchain[openai]" langchain-chroma langchain-text-splitters python-dotenv pypdf


## 2. API key y modelos

No conviene hardcodear claves en notebooks que se comparten con alumnos. Usamos variables de entorno o un archivo `.env`.

Ejemplo de `.env`:

```text
OPENAI_API_KEY=tu_api_key
OPENAI_MODEL=gpt-6-luna
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
```

Para clase conviene un modelo de bajo costo/latencia para chat y `text-embedding-3-small` para embeddings.


In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

print("Chat model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBEDDING_MODEL)


## 3. Primer objeto LangChain: chat model

La forma recomendada en LangChain v1 es `init_chat_model`. El equivalente específico de OpenAI es `ChatOpenAI`.

La llamada básica se hace con `.invoke()`.


In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(OPENAI_MODEL, temperature=0.2)

# Equivalente explícito:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0.2)

respuesta = llm.invoke("Explicá RAG en una frase simple para estudiantes de programación.")
print(respuesta.content)


## 4. Mini-chain con LCEL

LCEL permite componer piezas con `|`:

```text
prompt -> modelo -> parser
```

En v1 esto reemplaza a `LLMChain` (que vive en `langchain-classic`) para **pipelines de un solo paso**. No es un agente: no hay loop ni tools.


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = StrOutputParser()

prompt_intro = ChatPromptTemplate.from_messages([
    ("system", "Sos un profesor claro y práctico."),
    ("human", "Explicá el concepto de {tema} con una analogía simple."),
])

chain_intro = prompt_intro | llm | parser

print(chain_intro.invoke({"tema": "embeddings"}))


## 5. Documentos de ejemplo

Para entender RAG no necesitamos empezar con PDFs. Primero creamos documentos con metadatos.

Cada `Document` tiene:

- `page_content`: texto.
- `metadata`: información útil para filtrar o citar la fuente.
- `id`: identificador opcional (en producción conviene setearlo para poder actualizar o borrar chunks).


In [ ]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content=(
            "La empresa FicticiaBot ofrece tres planes: Básico, Pro y Enterprise. "
            "El plan Básico incluye 1.000 consultas mensuales y soporte por email."
        ),
        metadata={"source": "politicas_planes.md", "section": "planes"},
    ),
    Document(
        page_content=(
            "El plan Pro incluye 20.000 consultas mensuales, soporte prioritario y acceso a métricas. "
            "Tiene un límite de 5 usuarios por workspace."
        ),
        metadata={"source": "politicas_planes.md", "section": "planes"},
    ),
    Document(
        page_content=(
            "El plan Enterprise incluye consultas personalizadas, SSO, auditoría avanzada y soporte dedicado. "
            "Los límites se acuerdan por contrato."
        ),
        metadata={"source": "politicas_enterprise.md", "section": "enterprise"},
    ),
    Document(
        page_content=(
            "La política de privacidad establece que los datos de clientes no se usan para entrenar modelos externos. "
            "Los logs se retienen durante 30 días salvo acuerdo contractual diferente."
        ),
        metadata={"source": "privacidad.md", "section": "datos"},
    ),
    Document(
        page_content=(
            "Para solicitar baja del servicio, el cliente debe enviar una solicitud desde el panel de administración. "
            "La baja se procesa dentro de los 5 días hábiles."
        ),
        metadata={"source": "soporte.md", "section": "bajas"},
    ),
]

print("Cantidad de documentos:", len(docs))
print(docs[0])


## 6. Split de documentos en chunks

Los modelos tienen una ventana de contexto limitada y los documentos reales suelen ser largos. Por eso se dividen en fragmentos.

Parámetros importantes:

- `chunk_size`: tamaño aproximado del fragmento.
- `chunk_overlap`: solapamiento entre fragmentos para no cortar ideas importantes.

En producción, estos valores se ajustan según el tipo de documento y la calidad de recuperación.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=60,
)

splits = splitter.split_documents(docs)

print("Chunks generados:", len(splits))
for i, chunk in enumerate(splits[:3], start=1):
    print("--- chunk", i)
    print(chunk.page_content)
    print(chunk.metadata)




## 7. Embeddings

Un embedding convierte texto en un vector numérico. Textos semánticamente parecidos quedan cerca en el espacio vectorial.

En RAG se usan embeddings para buscar los chunks más parecidos a la pregunta del usuario.

En v1 el punto de entrada unificado es `init_embeddings`. El equivalente específico de OpenAI es `OpenAIEmbeddings`.


In [ ]:
from langchain.embeddings import init_embeddings

embeddings = init_embeddings(f"openai:{OPENAI_EMBEDDING_MODEL}")

# Equivalente explícito:
# from langchain_openai import OpenAIEmbeddings
# embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

vector = embeddings.embed_query("¿Qué incluye el plan Pro?")

print("Dimensión del vector:", len(vector))
print("Primeros 5 valores:", vector[:5])



## 8. Crear una base vectorial local con Chroma

Usamos `Chroma` desde `langchain_chroma`. Con `persist_directory` los vectores se guardan en disco; no hace falta llamar a `.persist()` (esa API vieja ya no existe).

Si re-ejecutamos `from_documents` sobre el mismo directorio, Chroma **agrega** chunks y duplica la colección. Por eso borramos la carpeta antes de indexar en clase.



In [ ]:
import shutil
from pathlib import Path

from langchain_chroma import Chroma

CHROMA_DIR = "./chroma_rag_demo"
COLLECTION_NAME = "curso_rag_basico"

if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
)

print("Vector store creado en:", Path(CHROMA_DIR).resolve())
print("Cantidad de chunks:", len(vectorstore.get()["ids"]))

## 9. Búsqueda semántica simple

Antes de usar un LLM, probamos si el buscador encuentra chunks relevantes.


In [ ]:
query = "¿Cuántos usuarios permite el plan Pro?"
results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"Resultado {i}")
    print("Fuente:", doc.metadata)
    print(doc.page_content)
    print()


## 10. Convertir Chroma en retriever

Un `retriever` recibe una pregunta y devuelve documentos relevantes. Es un `Runnable`, así que se usa con `.invoke()`.

Lo vamos a usar de dos formas: en el RAG 2-step (siempre recupera) y más adelante como tool de un agente (el modelo decide cuándo buscar).


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

retrieved_docs = retriever.invoke("¿Qué pasa con los datos de clientes?")

for doc in retrieved_docs:
    print(doc.metadata, "->", doc.page_content[:120], "...")


## 11. RAG 2-step: recuperar y después generar

En LangChain v1 el patrón recomendado para un FAQ / bot de documentos es **explícito**:

1. Pregunta del usuario.
2. `retriever.invoke(...)` siempre corre primero.
3. Armamos el prompt con ese contexto.
4. El modelo genera.
5. Parseamos a texto.

El prompt le pide al modelo que no invente si la respuesta no está en el contexto.

LCEL (`prompt | llm | parser`) queda para el paso lineal de generación, no para orquestar la app.


In [ ]:
def format_docs(documents):
    return "\n\n".join(
        f"Fuente: {doc.metadata.get('source')} | Sección: {doc.metadata.get('section')}\n{doc.page_content}"
        for doc in documents
    )

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Sos un asistente de soporte. Respondé usando únicamente el contexto dado. "
        "Si la respuesta no está en el contexto, decí: 'No encuentro esa información en la base de conocimiento'.",
    ),
    (
        "human",
        "Contexto:\n{context}\n\nPregunta: {question}",
    ),
])


def answer_rag(question: str) -> str:
    retrieved = retriever.invoke(question)
    context = format_docs(retrieved)
    return (rag_prompt | llm | parser).invoke({
        "context": context,
        "question": question,
    })


print(answer_rag("¿Qué incluye el plan Enterprise?"))


## 12. La misma pipeline con LCEL

Si querés un único `.invoke()` sobre la pregunta, LCEL compacta el 2-step. Es el mismo flujo: retriever → formato → prompt → modelo → parser.

Conviene cuando el grafo es lineal y conocido. Si el modelo tiene que **decidir** si busca, eso ya no es esta chain: es un agente.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | parser
)

print(rag_chain.invoke("¿Qué incluye el plan Enterprise?"))

## 13. RAG devolviendo respuesta + fuentes

En muchas apps no alcanza con responder: también queremos mostrar qué documentos fueron usados.

Por eso el 2-step conviene: la recuperación queda en una variable, no escondida dentro de la chain.


In [ ]:
def answer_with_sources(question: str):
    retrieved = retriever.invoke(question)
    context = format_docs(retrieved)
    answer = (rag_prompt | llm | parser).invoke({
        "context": context,
        "question": question,
    })
    sources = []
    for doc in retrieved:
        item = {
            "source": doc.metadata.get("source"),
            "section": doc.metadata.get("section"),
        }
        if item not in sources:
            sources.append(item)
    return {"answer": answer, "sources": sources}


result = answer_with_sources("¿Cuánto tardan en procesar la baja del servicio?")
print(result["answer"])
print("\nFuentes:")
for source in result["sources"]:
    print("-", source)



## 14. Pregunta fuera de la base

Una buena demo de RAG debe mostrar qué pasa cuando la información no está en los documentos.


In [ ]:
print(answer_rag("¿Cuál es el precio exacto del plan Pro en dólares?"))


## 15. Filtros por metadata

Chroma permite buscar usando metadatos. Esto es útil cuando tenemos secciones, clientes, fechas, permisos o tipos de documento.

Ejemplo: buscar solamente en documentos de la sección `planes`.


In [ ]:
results_filtered = vectorstore.similarity_search(
    "¿Qué incluye Enterprise?",
    k=3,
    filter={"section": "planes"},
)

for doc in results_filtered:
    print(doc.metadata)
    print(doc.page_content)
    print()


## 16. Cargar PDFs propios (opcional)

> Ejecutar solo si tenés archivos en la carpeta `data/`.

La guía actual de LangChain arma `Document` con `pypdf.PdfReader`. No uses `PyPDFLoader` de `langchain-community`: ese paquete está sunset.


In [ ]:
# Opcional: cargar PDFs desde una carpeta data/
# import pypdf
# from pathlib import Path
#
# pdf_docs = []
# for pdf_path in Path("data").glob("*.pdf"):
#     reader = pypdf.PdfReader(pdf_path)
#     for i, page in enumerate(reader.pages):
#         pdf_docs.append(
#             Document(
#                 page_content=page.extract_text() or "",
#                 metadata={"source": pdf_path.name, "page": i},
#             )
#         )
#
# pdf_splits = splitter.split_documents(pdf_docs)
#
# pdf_dir = Path("./chroma_pdfs")
# if pdf_dir.exists():
#     shutil.rmtree(pdf_dir)
#
# pdf_vectorstore = Chroma.from_documents(
#     documents=pdf_splits,
#     embedding=embeddings,
#     collection_name="mis_pdfs",
#     persist_directory=str(pdf_dir),
# )
#
# print("PDF chunks:", len(pdf_splits))
# print("Indexados:", len(pdf_vectorstore.get()["ids"]))


## 17. Hacia RAG agentic

En el 2-step el retriever **siempre** corre. En un agente, el modelo decide si necesita buscar.

Eso es el puente con clase 3: el retriever se envuelve en un `@tool` y se lo pasamos a `create_agent`.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def buscar_conocimiento(query: str) -> str:
    """Busca políticas, planes y soporte en la base de conocimiento de FicticiaBot."""
    retrieved = retriever.invoke(query)
    if not retrieved:
        return "No encontré documentos relevantes."
    return format_docs(retrieved)


rag_agent = create_agent(
    model=llm,
    tools=[buscar_conocimiento],
    system_prompt=(
        "Sos un asistente de soporte de FicticiaBot. "
        "Usá buscar_conocimiento antes de responder preguntas sobre el producto. "
        "Si la herramienta no trae la información, decí que no está en la base. "
        "No inventes precios ni condiciones."
    ),
)

result = rag_agent.invoke({
    "messages": [
        {"role": "user", "content": "¿El plan Pro permite más de 5 usuarios?"}
    ]
})
print(result["messages"][-1].content)

## Resumen

Patrón 2-step para un chatbot sobre documentos propios:

```text
loader -> splitter -> embeddings -> vector store -> retriever -> prompt -> LLM
```

En clase vimos tres capas:

- **Recuperación explícita** (`answer_rag`): fácil de debuggear y de devolver fuentes.
- **LCEL** (`prompt | modelo | parser`): compacta el paso lineal de generación.
- **`create_agent`**: el modelo decide si llama al retriever. Ahí RAG deja de ser un pipeline fijo.

Mejoras típicas después de este notebook:

- cargar PDFs, HTML, Markdown o bases de datos;
- mejorar chunking y agregar reranking;
- filtrar por permisos de usuario;
- persistir conversación con un checkpointer (`thread_id`);
- evaluar retrieval y calidad de respuesta.
